In [4]:
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo 'deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main' | tee /etc/apt/sources.list.d/google-chrome.list
!apt-get update
!apt-get install google-chrome-stable

OK
deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main
Hit:1 http://dl.google.com/linux/chrome/deb stable InRelease
Get:2 https://dl.google.com/linux/chrome-stable/deb stable InRelease [1,825 B]
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 https://cli.github.com/packages stable InRelease
Get:5 https://dl.google.com/linux/chrome-stable/deb stable/main amd64 Packages [1,219 B]
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Fetched 3,044 B in 1s (2,254 B/s)
Reading package lists... Done
W: http://dl.google.com/linux/chrome/deb/di

In [5]:
!pip install selenium beautifulsoup4 webdriver-manager

In [22]:
import time
import datetime
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# --- Step 2: HTMLから株価データを抽出する関数の作成 ---
def extract_stock_data(html):
    soup = BeautifulSoup(html, 'html.parser')
    try:
        # 日付の取得
        date_el = soup.select_one(".ChartHeader_date__fy3_Y")
        if not date_el: return None
        date_val = date_el.text.strip().replace('年', '-').replace('月', '-').replace('日', '')

        # 4つのデータ（始値・高値・安値・終値）を順番に取得
        # 全ての .ChartHeader_value__thRnp を取得し、その中から数値を探す
        value_elements = soup.select(".ChartHeader_value__thRnp")

        if len(value_elements) < 4:
            return None

        # 最初の4つをリストにする
        prices = [el.text.replace(',', '') for el in value_elements[:4]]

        return [date_val] + prices
    except:
        return None

# --- Step 3: Webページへアクセスし、データを抽出する関数の作成 ---
def get_stock_values(driver, url):
    """Seleniumを使用してグラフを走査し、株価データを全件取得する"""
    driver.get(url)
    wait = WebDriverWait(driver, 20)

    # グラフのコンテナが表示されるまで待機
    chart_container = wait.until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "div.Chart_container__2Be7S"))
    )

    # チャートのiframeに切り替え
    chart_iframe = chart_container.find_element(By.TAG_NAME, "iframe")
    driver.switch_to.frame(chart_iframe)
    time.sleep(5) # グラフ描画待機

    # svg要素を見つけて最大のもの（チャート本体）を特定
    svgs = driver.find_elements(By.TAG_NAME, "svg")
    visible_svgs = [svg for svg in svgs if svg.size["width"] > 0 and svg.size["height"] > 0]
    chart = max(visible_svgs, key=lambda svg: svg.size["width"] * svg.size["height"])

    # マウス操作の準備
    actions = ActionChains(driver)
    actions.move_to_element(chart).perform()
    chart_width = chart.size["width"]

    # マウスをグラフの右端に移動（中央から幅の半分だけ右へ）
    actions.move_by_offset(chart_width / 2, 0).perform()
    all_data = []

    # 1pxずつ左に移動しながらデータを取得
    for _ in range(int(chart_width)):
        actions.move_by_offset(-1, 0).perform()
        time.sleep(0.1) # 描画のために少し増やす

        # ラグ対策のためページソースを再取得
        current_html = driver.page_source
        data = extract_stock_data(current_html)

        # 重複を避けてリストに追加
        if data and data not in all_data:
            all_data.append(data)

    driver.switch_to.default_content()
    return all_data

# --- Step 4: mainコードの記述 ---
# Chromeの設定
chrome_options = Options()
chrome_options.add_argument('--headless=new')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--window-size=1920,1080')

# ドライバーの起動
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

url = "https://www.nikkei.com/markets/worldidx/chart/nk225/?type=6month"

start_time = time.time()
results = get_stock_values(driver, url)
end_time = time.time()

# 結果表示
print(f"スクレイピング所要時間: {end_time - start_time:.2f} 秒")
for row in results:
    print(f"日付：{row[0]},始値：{row[1]},高値：{row[2]},安値：{row[3]},,高値：{row[2]},安値：{row[3]},終値：{row[4]}")

driver.quit()

スクレイピング所要時間: 275.05 秒
日付：2026/06/26,始値：71587.71,高値：71786.28,安値：68639.84,,高値：71786.28,安値：68639.84,終値：69360.88
日付：2026/06/25,始値：70114.09,高値：72594.22,安値：69982.67,,高値：72594.22,安値：69982.67,終値：72366.34
日付：2026/06/24,始値：69615.08,高値：70218.71,安値：68461.1,,高値：70218.71,安値：68461.1,終値：69174.97
日付：2026/06/23,始値：72404.37,高値：72618.44,安値：69788.38,,高値：72618.44,安値：69788.38,終値：69788.38
日付：2026/06/22,始値：71067.15,高値：72831.73,安値：71009.52,,高値：72831.73,安値：71009.52,終値：72353.96
日付：2026/06/19,始値：71551.03,高値：71952.99,安値：70517.98,,高値：71952.99,安値：70517.98,終値：71250.06
日付：2026/06/18,始値：70163.71,高値：71398.58,安値：70092.94,,高値：71398.58,安値：70092.94,終値：71053.49
日付：2026/06/17,始値：69005.88,高値：70125.75,安値：68985.63,,高値：70125.75,安値：68985.63,終値：69902.25
日付：2026/06/16,始値：69288.91,高値：70020.68,安値：69095.67,,高値：70020.68,安値：69095.67,終値：69404.5
日付：2026/06/15,始値：66783.22,高値：69682.23,安値：66783.22,,高値：69682.23,安値：66783.22,終値：69317.5
日付：2026/06/12,始値：65176.23,高値：67065.94,安値：64998.11,,高値：67065.94,安値：64998.11,終値：66020.04
日付：2026/06/11,始値：63329.17